# Plain implementation of YOLOv12 trained on our custom dataset (RGB)
Created following this tutorial https://blog.roboflow.com/train-yolov12-model/

### Installing required packages

In [3]:
!git clone https://github.com/sunsmarterjie/yolov12
%cd yolov12
%pip install roboflow supervision flash-attn --upgrade -q
%pip install -r requirements.txt
%pip install -e .
%pip install --upgrade flash-attn

# Not in tutorial but necesarry:
%pip install huggingface_hub ultralytics

Cloning into 'yolov12'...
remote: Enumerating objects: 968, done.
remote: Counting objects: 100% (246/246), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 968 (delta 219), reused 199 (delta 189), pack-reused 722 (from 1)
Receiving objects: 100% (968/968), 1.60 MiB | 6.64 MiB/s, done.
Resolving deltas: 100% (460/460), done.
/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov12
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.idi.ntnu.no
ERROR: flash_attn-2.7.3+cu11torch2.2cxx11abiFALSE-cp311-cp311-linux_x86_64.whl is not a supported wheel on this platform.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.idi.ntnu.no
Obtaining file:///home/omtalmo/Olaf_TTK4

### Loading dataset
The data is structured like this: I have one folder called "rgb". Inside the folder is a data.yaml file, a folder called "images" and a folder called "labels". The folder "images" contains the subfolders "test", "train" and "valid", which all contain png files. The folder "labels" contain the subfolders "train" and "valid", which both contain txt files. 

In [8]:
dataset_path = "/home/omtalmo/Olaf_TTK4265/Poles/rgb"

In [ ]:
from ultralytics import YOLO

# Initialize YOLOv12 model (s = small version)
model = YOLO('yolov12s.yaml')  

# Train the model using RGB data
results = model.train(
    data=f'{dataset_path}/data.yaml',
    epochs=250
)

In [ ]:
import random
import supervision as sv  # Import the missing module
import cv2  # Ensure OpenCV is imported if used later


HOME = "home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/yolov12"
model = YOLO(f"/{HOME}/yolov12/runs/detect/train3/weights/best.pt")

ds = sv.DetectionDataset.from_yolo(
	images_directory_path=f"{dataset_path}/images/valid",
	annotations_directory_path=f"{dataset_path}/labels/valid",
	data_yaml_path=f"{dataset_path}/data.yaml"
)

image = random.choice(list(ds.images.keys()))
image = cv2.imread(image)

bounding_box_annotator = sv.BoundingBoxAnnotator()
label_annotator = sv.LabelAnnotator()

results = model(image)[0]
detections = sv.Detections.from_ultralytics(results).with_nms()

bounding_box_annotator = sv.BoundingBoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = bounding_box_annotator.annotate(
	scene=image, detections=detections)
annotated_image = label_annotator.annotate(
	scene=annotated_image, detections=detections)

sv.plot_image(annotated_image)

SupervisionWarnings: images is deprecated: `DetectionDataset.images` property is deprecated and will be removed in `supervision-0.26.0`. Iterate with `for path, image, annotation in dataset:` instead.


ValueError: No images found in the dataset. Check the dataset paths and structure.